In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import umap

In [2]:
data = pd.read_csv('Physical_Activity_Monitoring_unlabeled.csv')
data.shape

(534601, 53)

In [3]:
data.isna().sum()

,0
timestamp,0
handTemperature,4041
handAcc16_1,4041
handAcc16_2,4041
handAcc16_3,4041
handAcc6_1,4041
handAcc6_2,4041
handAcc6_3,4041
handGyro1,4041
handGyro2,4041


In [4]:
y = data['subject_id']
data = data.drop(columns=['subject_id', 'timestamp'])
data = data.interpolate(method='linear').fillna(method='bfill')

/tmp/ipykernel_11498/328721576.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.interpolate(method='linear').fillna(method='bfill')


In [5]:
scaler = StandardScaler()

In [6]:
df_scaled = data.copy()
df_scaled.iloc[:, :] = scaler.fit_transform(data)

In [7]:
df_scaled.isna().sum()

,0
handTemperature,0
handAcc16_1,0
handAcc16_2,0
handAcc16_3,0
handAcc6_1,0
handAcc6_2,0
handAcc6_3,0
handGyro1,0
handGyro2,0
handGyro3,0


In [8]:
len(y.unique())

8

In [15]:
pca = PCA(n_components=30)
df_pca = pca.fit_transform(df_scaled)

In [16]:
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
clusters = kmeans.fit_predict(df_pca)

In [17]:
df_result = pd.DataFrame(df_pca)
df_result["cluster"] = clusters

df_result.head()

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,cluster
0,5.825618,0.391503,-0.048351,-0.894547,-0.665255,0.549415,-0.953569,0.311919,1.818166,0.104333,...,-0.134425,-0.050059,-0.274222,0.237001,-0.206043,0.184014,0.098342,0.334695,0.202599,1
1,-0.679900,-3.897040,-1.758776,0.201861,0.640188,-0.526182,2.859060,-1.103071,-0.235493,-0.882236,...,0.281463,-0.379937,0.750791,0.720046,1.462264,-0.063235,0.182552,-0.471362,0.207227,3
2,0.041319,2.551187,-1.512075,1.191609,-0.617989,-1.908663,-0.519557,0.848197,0.584015,0.242471,...,1.213291,-0.373601,0.119912,0.377028,-0.234357,0.586569,0.559427,-0.436184,-0.231478,6
3,-3.705234,1.514374,-0.626194,1.003345,1.506819,-0.721592,0.159022,-2.255705,1.409545,2.257071,...,0.747797,-0.990634,-3.296590,-0.833973,0.701208,0.592099,-0.179717,-1.172463,-0.280693,6
4,0.515698,-0.556662,1.158513,-2.125274,-0.285394,0.992734,0.815029,0.601382,0.506297,-0.111657,...,0.330296,-0.374868,-0.001728,0.259005,0.193064,-0.505001,-0.281831,-0.379505,0.059853,5


In [18]:
mapping = {}
maxi = 1
new_vals = list()

for val in df_result['cluster']:
  if val not in mapping:
    mapping[val] = maxi
    maxi += 1
  new_vals.append(mapping[val])
df_result['final_cluster'] = new_vals
df_result.head()

answer = df_result['final_cluster']

submission = pd.DataFrame({
    'Index': answer.index,
    'activityID': answer.values
})

submission.to_csv('submission.csv', index=False)

submission.shape

(534601, 2)